# Balanceamento do Dataset TFRecord — Fotovoltaica

Diagnóstico: 1029 tfrecords de treino, amostragem de 20000 patches mostrou
15% positivos (com painel) / 85% negativos (só background), razão neg/pos = 5.67.

**O que este notebook faz:**

- **Parte 1** — audita **todos** os patches de **todos** os tfrecords da pasta `train/`
  (contagem exata, não amostrada) e move para uma pasta de quarentena os arquivos que
  não têm **nenhum** patch com painel (100% background).
- **Parte 2** — a partir da razão neg/pos resultante, calcula quantos arquivos da
  quarentena precisam voltar para `train/` para atingir uma razão-alvo configurável,
  e os move de volta.

Reaproveita as funções de leitura/estatística de `visualize_tfrecord_patches_colab.ipynb`.

**Nota:** o notebook de treino (`train_fotovoltaica_colab_v2.ipynb`) já possui um filtro
`filter_empty_patches` + `EMPTY_PATCH_KEEP_RATE` que descarta patches vazios *durante* o
treino (em memória). Este notebook resolve um problema diferente: reduz o volume de
dados **em disco** (menos arquivos para o `tf.data` ler do Drive a cada época), então
não é necessário — nem desejável — zerar a razão neg/pos aqui; basta trazê-la para um
patamar moderado (ex.: 2–4:1) e deixar o filtro em memória fazer o ajuste fino.

## 1. Autenticação e montagem do Drive

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)
print('Drive montado.')


## 2. Imports

In [ ]:
import os
import glob
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
print(f'TensorFlow {tf.__version__}')


## 3. Configurações — ajuste os caminhos aqui

In [ ]:
# Pasta de treino a auditar (mesma usada pelo train_fotovoltaica_colab_v2.ipynb)
DATASET_ROOT   = '/content/drive/MyDrive/DS_FV_tfs_scaled_v3'
TRAIN_DIR      = os.path.join(DATASET_ROOT, 'train')

# Pasta de quarentena — recebe os tfrecords 100% background (sem nenhum painel)
QUARANTINE_DIR = os.path.join(DATASET_ROOT, 'train_sem_alerta')
os.makedirs(QUARANTINE_DIR, exist_ok=True)

# Onde salvar o manifesto (CSV) com a auditoria por arquivo — permite reexecutar
# a Parte 2 sem reler todos os tfrecords novamente.
MANIFEST_PATH = os.path.join(DATASET_ROOT, 'manifest_balanceamento_train.csv')

# Bandas e label — devem coincidir com o script de exportação GEE / notebook de treino
BANDS_LIST     = ['blue', 'green', 'red', 'pvi', 'pvpi']
LABEL_KEY      = 'label'
PATCH_SIZE     = 256    # tamanho final após crop
RAW_PATCH_SIZE = 257    # GEE exporta 257×257 (kernel rectangle(128,128))

# Razão-alvo negativo:positivo desejada em disco na pasta train/ após o rebalanceamento.
# Ex.: 3.0 → para cada patch positivo, manter ~3 patches negativos.
TARGET_NEG_POS_RATIO = 3.0

# Seed para a seleção aleatória de arquivos a devolver na Parte 2 (reprodutibilidade)
RANDOM_SEED = 42

print(f'TRAIN_DIR      : {TRAIN_DIR}')
print(f'QUARANTINE_DIR : {QUARANTINE_DIR}')
print(f'MANIFEST_PATH  : {MANIFEST_PATH}')


## 4. Funções de leitura do TFRecord

Reaproveitadas de `visualize_tfrecord_patches_colab.ipynb`, mas com uma versão "label-only" — parseia **apenas** a banda `label` (ignora as 5 bandas espectrais) para tornar a auditoria de 1029 arquivos bem mais rápida.

In [ ]:
# Descrição completa (usada só se quiser inspecionar imagens depois)
FEATURE_DESCRIPTION = {
    key: tf.io.FixedLenFeature([RAW_PATCH_SIZE, RAW_PATCH_SIZE], tf.float32)
    for key in BANDS_LIST + [LABEL_KEY]
}

# Descrição reduzida — só o label. tf.io.parse_single_example aceita um
# subconjunto das chaves serializadas, então isso evita reconstruir os
# tensores das 5 bandas espectrais durante a auditoria.
LABEL_FEATURE_DESCRIPTION = {
    LABEL_KEY: tf.io.FixedLenFeature([RAW_PATCH_SIZE, RAW_PATCH_SIZE], tf.float32)
}


def parse_label_only(example_proto):
    parsed = tf.io.parse_single_example(example_proto, LABEL_FEATURE_DESCRIPTION)
    return tf.slice(parsed[LABEL_KEY], [0, 0], [PATCH_SIZE, PATCH_SIZE])


def audit_tfrecord_file(path):
    """Lê todos os patches de um tfrecord e conta positivos/negativos."""
    compression = 'GZIP' if path.endswith('.gz') else ''
    ds = tf.data.TFRecordDataset([path], compression_type=compression)
    ds = ds.map(parse_label_only, num_parallel_calls=tf.data.AUTOTUNE)

    n_total = 0
    n_pos   = 0
    for label in ds:
        n_total += 1
        if tf.reduce_sum(label) > 0:
            n_pos += 1

    return n_total, n_pos


print('Funções de leitura definidas.')


## 5. Parte 1 — Auditoria exata: patches por tfrecord

Percorre **todos** os arquivos de `TRAIN_DIR`, conta patches positivos/negativos de cada um, e marca `has_alert = False` para os arquivos 100% background.

In [ ]:
all_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*.tfrecord.gz')))
if not all_files:
    all_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*.tfrecord')))
print(f'Arquivos encontrados em train/: {len(all_files)}')

records = []
for i, path in enumerate(all_files):
    n_total, n_pos = audit_tfrecord_file(path)
    n_neg = n_total - n_pos
    records.append({
        'file': path,
        'basename': os.path.basename(path),
        'n_total': n_total,
        'n_pos': n_pos,
        'n_neg': n_neg,
        'has_alert': n_pos > 0,
        'status': 'kept' if n_pos > 0 else 'quarantined',
    })
    if (i + 1) % 100 == 0 or (i + 1) == len(all_files):
        print(f'  auditados {i + 1}/{len(all_files)} arquivos...')

manifest = pd.DataFrame.from_records(records)
manifest.to_csv(MANIFEST_PATH, index=False)

n_files_no_alert = int((~manifest['has_alert']).sum())
n_files_with_alert = int(manifest['has_alert'].sum())
total_pos = int(manifest['n_pos'].sum())
total_neg = int(manifest['n_neg'].sum())

print(f'\n{"="*50}')
print(f'  Arquivos auditados        : {len(manifest)}')
print(f'  Arquivos SEM nenhum alerta: {n_files_no_alert}')
print(f'  Arquivos COM algum alerta : {n_files_with_alert}')
print(f'  Patches positivos (total) : {total_pos}')
print(f'  Patches negativos (total) : {total_neg}')
print(f'  Razão neg/pos (exata)     : {total_neg / (total_pos + 1e-9):.2f}')
print(f'{"="*50}')
print(f'\nManifesto salvo em: {MANIFEST_PATH}')


## 6. Mover para quarentena os arquivos sem nenhum patch com alerta

Move fisicamente para `QUARANTINE_DIR` todo tfrecord com `has_alert == False`. O manifesto é atualizado com o novo caminho de cada arquivo.

In [ ]:
moved = 0
for idx, row in manifest[~manifest['has_alert']].iterrows():
    src = row['file']
    if not os.path.exists(src):
        continue
    dst = os.path.join(QUARANTINE_DIR, row['basename'])
    shutil.move(src, dst)
    manifest.at[idx, 'file'] = dst
    moved += 1

manifest.to_csv(MANIFEST_PATH, index=False)
print(f'Arquivos movidos para quarentena: {moved}')
print(f'Arquivos restantes em train/    : {len(glob.glob(os.path.join(TRAIN_DIR, "*.tfrecord.gz")))}')


## 7. Estatística exata do dataset de treino após a remoção

Calculada direto do manifesto (contagem exata da Parte 1, não amostrada) — soma apenas os arquivos com `status == 'kept'`.

In [ ]:
kept = manifest[manifest['status'] == 'kept']
quarantined = manifest[manifest['status'] == 'quarantined']

pos_kept = int(kept['n_pos'].sum())
neg_kept = int(kept['n_neg'].sum())

print(f'{"="*50}')
print(f'  Arquivos mantidos em train/  : {len(kept)}')
print(f'  Arquivos em quarentena       : {len(quarantined)}')
print(f'  Patches positivos (train/)   : {pos_kept}')
print(f'  Patches negativos (train/)   : {neg_kept}')
print(f'  Razão neg/pos (train/)       : {neg_kept / (pos_kept + 1e-9):.2f}')
print(f'  Patches negativos em quarent.: {int(quarantined["n_neg"].sum())}  ({len(quarantined)} arquivo(s))')
print(f'{"="*50}')


## 8. Parte 2 — Reintroduzir arquivos da quarentena até a razão-alvo

Seleciona aleatoriamente (com seed fixa) arquivos da quarentena e os devolve para `train/` até que a razão neg/pos se aproxime de `TARGET_NEG_POS_RATIO`, sem ultrapassar muito o alvo (seleção gulosa, um arquivo por vez).

In [ ]:
random.seed(RANDOM_SEED)

desired_total_neg     = TARGET_NEG_POS_RATIO * pos_kept
additional_neg_needed = desired_total_neg - neg_kept

print(f'Positivos em train/ (fixo)       : {pos_kept}')
print(f'Negativos em train/ (atual)      : {neg_kept}')
print(f'Razão-alvo neg/pos               : {TARGET_NEG_POS_RATIO:.2f}')
print(f'Negativos necessários para o alvo: {desired_total_neg:.0f}')
print(f'Negativos a trazer da quarentena : {max(0, additional_neg_needed):.0f}')

restored_idx = []
if additional_neg_needed <= 0:
    print('\nRazão atual já atende (ou excede) o alvo — nenhum arquivo será devolvido.')
else:
    candidates = quarantined.index.tolist()
    random.shuffle(candidates)

    acc_neg = 0
    for idx in candidates:
        if acc_neg >= additional_neg_needed:
            break
        acc_neg += int(manifest.at[idx, 'n_neg'])
        restored_idx.append(idx)

    print(f'\nArquivos selecionados para retornar: {len(restored_idx)}')
    print(f'Negativos que serão adicionados    : {acc_neg}')


In [ ]:
moved_back = 0
for idx in restored_idx:
    src = manifest.at[idx, 'file']
    if not os.path.exists(src):
        continue
    dst = os.path.join(TRAIN_DIR, manifest.at[idx, 'basename'])
    shutil.move(src, dst)
    manifest.at[idx, 'file'] = dst
    manifest.at[idx, 'status'] = 'restored'
    moved_back += 1

manifest.to_csv(MANIFEST_PATH, index=False)
print(f'Arquivos devolvidos para train/: {moved_back}')


## 9. Estatística final (após rebalanceamento)

In [ ]:
final_kept = manifest[manifest['status'].isin(['kept', 'restored'])]
final_pos  = int(final_kept['n_pos'].sum())
final_neg  = int(final_kept['n_neg'].sum())

print(f'{"="*50}')
print(f'  Arquivos em train/ (final)  : {len(final_kept)}')
print(f'  Arquivos ainda em quarentena: {(manifest["status"] == "quarantined").sum()}')
print(f'  Patches positivos           : {final_pos}')
print(f'  Patches negativos           : {final_neg}')
print(f'  Razão neg/pos final         : {final_neg / (final_pos + 1e-9):.2f}')
print(f'{"="*50}')

manifest['status'].value_counts()


## 10. (Opcional) Conferência visual — amostra de patches de train/ pós-rebalanceamento

Reaproveita a rotina de estatística amostrada de `visualize_tfrecord_patches_colab.ipynb` como checagem visual rápida (histograma) — a decisão da Parte 2 já usou a contagem exata.

In [ ]:
N_STATS_PATCHES = 5000

def to_label_array(example_proto):
    return parse_label_only(example_proto)

final_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*.tfrecord.gz')))
ds_check = (tf.data.TFRecordDataset(final_files, compression_type='GZIP',
                                     num_parallel_reads=tf.data.AUTOTUNE)
            .shuffle(buffer_size=1000, reshuffle_each_iteration=False)
            .map(to_label_array, num_parallel_calls=tf.data.AUTOTUNE)
            .take(N_STATS_PATCHES))

panel_pixels = []
for label in ds_check:
    panel_pixels.append(int(tf.reduce_sum(label).numpy()))

panel_pixels = np.array(panel_pixels)
n_pos_sample = int((panel_pixels > 0).sum())
n_total_sample = len(panel_pixels)

print(f'Amostra: {n_total_sample} patches | '
      f'{n_pos_sample} positivos ({100*n_pos_sample/n_total_sample:.1f}%) | '
      f'razão neg/pos ≈ {(n_total_sample - n_pos_sample) / (n_pos_sample + 1e-9):.2f}')

plt.figure(figsize=(6, 4))
plt.hist(panel_pixels, bins=50, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linewidth=1.5, linestyle='--', label='0 px (background)')
plt.xlabel('Pixels de painel no patch')
plt.ylabel('Número de patches')
plt.title(f'train/ pós-rebalanceamento (N={n_total_sample})')
plt.legend()
plt.tight_layout()
plt.show()
